<a href="https://colab.research.google.com/github/finningr/late_and_ungraded_work/blob/main/weekly_two.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from itertools import product, combinations
from collections import deque
from time import perf_counter
import gc
import pandas as pd

In [ ]:
def all_positions(n):
    return ["".join(p) for p in product("TF.", repeat=n)]


def positions_with_counts(n, t, f):
    positions = []
    for toad_spots in combinations(range(n), t):
        remaining = [i for i in range(n) if i not in toad_spots]
        for frog_spots in combinations(remaining, f):
            board = ["."] * n
            for i in toad_spots:
                board[i] = "T"
            for i in frog_spots:
                board[i] = "F"
            positions.append("".join(board))
    return positions

In [ ]:
def time_function_one(start_n=3, max_seconds=1800, max_n=12):
    rows = []
    n = start_n

    while True:
        try:
            start = perf_counter()
            positions = all_positions(n)
            seconds = perf_counter() - start

            rows.append({"n": n, "size": len(positions), "seconds": seconds})

            del positions
            gc.collect()

            if seconds >= max_seconds:
                break

            if max_n is not None and n >= max_n:
                break

            n += 1

        except MemoryError:
            rows.append({"n": n, "size": "MemoryError", "seconds": None})
            break

    return pd.DataFrame(rows)


def time_function_two(t, f, start_n=3, max_seconds=1800, max_n=20):
    rows = []
    n = start_n

    while True:
        try:
            start = perf_counter()
            positions = positions_with_counts(n, t, f)
            seconds = perf_counter() - start

            rows.append({"n": n, "size": len(positions), "seconds": seconds})

            del positions
            gc.collect()

            if seconds >= max_seconds:
                break

            if max_n is not None and n >= max_n:
                break

            n += 1

        except MemoryError:
            rows.append({"n": n, "size": "MemoryError", "seconds": None})
            break

    return pd.DataFrame(rows)

In [ ]:
def moves(position):
    result = []
    n = len(position)

    for i in range(n):
        if position[i] == "T":
            if i + 1 < n and position[i + 1] == ".":
                board = list(position)
                board[i], board[i + 1] = ".", "T"
                result.append("".join(board))

            if i + 2 < n and position[i + 1] == "F" and position[i + 2] == ".":
                board = list(position)
                board[i], board[i + 2] = ".", "T"
                result.append("".join(board))

        if position[i] == "F":
            if i - 1 >= 0 and position[i - 1] == ".":
                board = list(position)
                board[i], board[i - 1] = ".", "F"
                result.append("".join(board))

            if i - 2 >= 0 and position[i - 1] == "T" and position[i - 2] == ".":
                board = list(position)
                board[i], board[i - 2] = ".", "F"
                result.append("".join(board))

    return result


def breadth_first_search(position):
    queue = deque([position])
    count = 0

    while queue:
        current = queue.popleft()
        count += 1

        for next_position in moves(current):
            queue.append(next_position)

    return count


def depth_first_search(position):
    stack = [position]
    count = 0

    while stack:
        current = stack.pop()
        count += 1

        for next_position in moves(current):
            stack.append(next_position)

    return count

In [ ]:
table_problem_1 = time_function_one()
table_problem_1

,n,size,seconds
0,3,27,0.000015
1,4,81,0.000038
2,5,243,0.000084
3,6,729,0.000239
4,7,2187,0.000644
5,8,6561,0.007836
6,9,19683,0.012375
7,10,59049,0.033725
8,11,177147,0.107688
9,12,531441,0.397424


In [ ]:
t = 2
f = 2

table_problem_2 = time_function_two(t, f)
table_problem_2

,n,size,seconds
0,3,0,0.000012
1,4,6,0.000041
2,5,30,0.000086
3,6,90,0.000148
4,7,210,0.000292
5,8,420,0.000455
6,9,756,0.002659
7,10,1260,0.001298
8,11,1980,0.001961
9,12,2970,0.002924


In [ ]:
start_position = "TTT.FFF"

bfs_nodes = breadth_first_search(start_position)
dfs_nodes = depth_first_search(start_position)

bfs_nodes, dfs_nodes

(101, 101)